# FIFA 21 Player Data — ELT Pipeline

End-to-end pipeline reading the raw FIFA 21 dataset, applying data cleaning and transformations, and producing analytical aggregations.

**Source:** `fifa21_raw_data.csv` via Unity Catalog Volume  
**Target:** `fifa21.gold.*`

---

## 1. Ingestion
Read the raw CSV from the Unity Catalog Volume. `multiLine=True` and `escape='"'` are required due to embedded newlines and quoted fields in the source file. Column names are sanitised immediately to comply with Delta Lake naming requirements.

In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp
import re

@dp.materialized_view(
    comment="Bronze layer: Raw FIFA 21 player data ingested from CSV with clean column names"
)
def bronze_raw():
    df = spark.read.csv("dbfs:/Volumes/fifa21/default/fifa21_volume/fifa21_raw_data.csv", header=True, inferSchema=True, multiLine=True, escape='"')
    
    # Fix column names immediately to comply with Delta Lake requirements. The column names had spaces between the words. This is not allowed in Delta Lake.
    for col_name in df.columns:
        clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        if clean_name != col_name:
            df = df.withColumnRenamed(col_name, clean_name)
    
    return df

## 2. Cleaning

Reads from `bronze_raw` and applies all transformations in a single materialized view.

**Transformations applied:**
- **2.1 Newline removal & column drops** — the multiline CSV parser introduces embedded `\n` characters in string columns. These are removed before any further processing. Columns with no analytical value (`photoUrl`, `playerUrl`, `LongName`) are dropped.
- **2.2 Contract parsing** — the `Team_Contract` column encodes three formats: `TeamName 2018 ~ 2023` (contract), `TeamName Jun 30, 2022 On Loan` (loan), and `NationalityFree` (free agent). Parsed into `TypeOfContract`, `TeamName`, `StartYear`, and `EndYear`.
- **2.3 Height & weight conversion** — height converted from feet/inches (e.g. `5'11"`) to metres; weight from pounds (e.g. `175lbs`) to kilograms.
- **2.4 Star rating columns** — `IR`, `W_F`, and `SM` contain a `★` symbol which is stripped before casting to integer.
- **2.5 Type conversions** — money columns (`Wage`, `Value`, `Release_Clause`) parsed from `€XK`/`€XM` format to integer. `Joined` parsed to year integer, `Hits` cast to integer.

In [0]:
@dp.materialized_view(
    comment="Silver layer: Cleaned and transformed FIFA 21 player data"
)
def silver_cleaned():
    # Read from bronze (already has clean column names)
    df = spark.read.table("bronze_raw")
    
    # 2.1: Remove embedded newline characters introduced by multiline CSV parsing
    string_cols = [
        "photoUrl", "LongName", "playerUrl", "Nationality", "Positions", "Name",
        "Team_Contract", "Height", "Weight", "foot", "BP", "Joined",
        "Loan_Date_End", "Value", "Wage", "Release_Clause", "W_F", "SM",
        "A_W", "D_W", "IR", "Hits"
    ]
    
    for col in string_cols:
        if col in df.columns:
            df = df.withColumn(col, F.regexp_replace(col, "\n", ""))
    
    # Remove not needed columns
    df1 = df.drop("photoUrl", "playerUrl", "LongName")
    
    # 2.2: Contract parsing
    df1 = df1.withColumn("TypeOfContract", 
        F.when(F.col("Team_Contract").contains("~"), "Contract")
        .when(F.col("Team_Contract").contains(","), "Loan")
        .otherwise("Free Agent"))
    
    df1 = df1.withColumn("TeamName", 
        F.when(F.col("TypeOfContract") == "Contract", F.split(F.col("Team_Contract"), r"\d")[0])
        .when(F.col("TypeOfContract") == "Loan", F.substring(F.col("Team_Contract"), 1, F.length(F.col("Team_Contract")) - 20))
        .otherwise("Nationality"))
    
    df1 = df1.withColumn("contract_years", 
        F.when(F.col("TypeOfContract") == "Contract", F.regexp_replace(F.regexp_extract("Team_Contract", r"(\d{4}\s*~\s*\d{4})", 1), " ", ""))
        .when(F.col("TypeOfContract") == "Loan", F.substring(F.col("Team_Contract"), F.length(F.col("Team_Contract")) - 11, 4))
        .otherwise("0"))
    
    df1 = df1.withColumn("StartYear", 
        F.when(F.col("TypeOfContract") == "Contract", F.expr("CAST(get(split(contract_years, '~'), 0) AS int)"))
        .otherwise(F.col("contract_years").cast("int")))
    
    df1 = df1.withColumn("EndYear", 
        F.when(F.col("TypeOfContract") == "Contract", F.expr("CAST(get(split(contract_years, '~'), 1) AS int)"))
        .when(F.col("TypeOfContract") == "Loan", F.substring(F.col("Loan_Date_End"), F.length(F.col("Loan_Date_End")) - 4, 5).cast("int"))
        .otherwise(F.col("contract_years").cast("int")))
    
    df1 = df1.drop("Team_Contract", "contract_years", "Loan_Date_End", 'BOV', 'BP')
    
    # 2.3: Height & weight conversion
    df2 = df1.withColumn("feet", F.split("Height", "'")[0].cast("int"))
    df2 = df2.withColumn("inches", F.regexp_replace(F.split("Height", "'")[1], '"', "").cast("int"))
    df2 = df2.withColumn("Height", F.round((F.col("feet") * 12 + F.col("inches")) * 0.0254, 1))
    df2 = df2.drop("feet", "inches")
    df2 = df2.withColumn("Weight", F.round(F.split("Weight", "lbs")[0].cast("int") * 0.453592, 0).cast("int"))
    
    # 2.4: Star rating columns
    df3 = df2.withColumn("IR", F.regexp_replace(F.col("IR"), '★', '').cast("int"))
    df3 = df3.withColumn("W_F", F.regexp_replace(F.col("W_F"), '★', '').cast("int"))
    df3 = df3.withColumn("SM", F.regexp_replace(F.col("SM"), '★', '').cast("int"))
    
    # 2.5: Type conversions
    df4 = df3.withColumn("Hits", F.col("Hits").cast("int"))
    
    money_cols = ["Wage", "Value", "Release_Clause"]
    for c in money_cols:
        df4 = df4.withColumn(c, F.expr(f"""
            CASE
                WHEN `{c}` LIKE '%M' THEN CAST(CAST(REGEXP_REPLACE(`{c}`, '[€M]', '') AS DOUBLE) * 1000000 AS INT)
                WHEN `{c}` LIKE '%K' THEN CAST(CAST(REGEXP_REPLACE(`{c}`, '[€K]', '') AS DOUBLE) * 1000 AS INT)
                ELSE CAST(REGEXP_REPLACE(`{c}`, '[€]', '') AS INT)
            END"""))
    
    df4 = df4.withColumn("Joined", F.substring(F.col("Joined"), F.length(F.col("Joined")) - 4, 5).cast("int"))
    
    return df4

---

## 3. Analysis

### 3.1 Wage vs Value
Scatter plot of weekly wage against market value. Saved to Gold for dashboard use.

In [0]:
@dp.materialized_view(
    comment="Gold: Wage vs Value scatter plot data for dashboard"
)
def gold_wage_value_scatter():
    df = spark.read.table("silver_cleaned")
    return df.select("Name", "Age", "Nationality", "Positions", "Wage", "Value").filter(
        F.col("Wage").isNotNull() & F.col("Value").isNotNull()
    )

### 3.2 Age distribution
Player count by age across the dataset.

In [0]:
@dp.materialized_view(
    comment="Gold: Player count by age distribution"
)
def gold_age_distribution():
    df = spark.read.table("silver_cleaned")
    return df.groupBy("Age").agg(F.count("*").alias("player_count")).orderBy("Age")

### 3.3 Wage distribution
Player count binned into €5K weekly wage bands. Highlights the skewed earning structure across the player pool.

In [0]:
@dp.materialized_view(
    comment="Gold: Player count by €5K wage bands"
)
def gold_wage_distribution():
    df = spark.read.table("silver_cleaned")
    wage_dist = df.filter(F.col("Wage").isNotNull() & (F.col("Wage") > 0))
    wage_dist = wage_dist.withColumn("wage_band", (F.floor(F.col("Wage") / 5000) * 5000).cast("int"))
    return wage_dist.groupBy("wage_band").agg(F.count("*").alias("player_count")).orderBy("wage_band")

### 3.4 Value distribution
Player count binned into €1M market value bands.

In [0]:
@dp.materialized_view(
    comment="Gold: Player count by €1M value bands"
)
def gold_value_distribution():
    df = spark.read.table("silver_cleaned")
    value_dist = df.filter(F.col("Value").isNotNull() & (F.col("Value") > 0))
    value_dist = value_dist.withColumn("value_band", (F.floor(F.col("Value") / 1000000) * 1000000).cast("int"))
    return value_dist.groupBy("value_band").agg(F.count("*").alias("player_count")).orderBy("value_band")

### 3.5 Long-stay players
Players who have been at their club for 10 or more years.

In [0]:
@dp.materialized_view(
    comment="Gold: Players with 10+ years at their club"
)
def gold_long_stay_players():
    df = spark.read.table("silver_cleaned")
    return df.filter(F.col("Joined") <= F.col("EndYear") - 10)

### 3.6 Position stats
Average wage, market value and overall rating by position. Players with multiple positions are counted once per position via explode.

In [0]:
@dp.materialized_view(
    comment="Gold: Average stats by position (exploded)"
)
def gold_position_stats():
    df = spark.read.table("silver_cleaned")
    position_stats = df.filter(F.col("Positions").isNotNull())
    position_stats = position_stats.withColumn("Position", F.explode(F.split(F.col("Positions"), " ")))
    position_stats = position_stats.filter(F.col("Position") != "")
    return position_stats.groupBy("Position").agg(
        F.count("*").alias("player_count"),
        F.round(F.avg("Wage"), 0).cast("int").alias("avg_wage"),
        F.round(F.avg("Value"), 0).cast("int").alias("avg_value"),
        F.round(F.avg("OVA"), 1).alias("avg_overall")
    ).orderBy("player_count", ascending=False)